# Oppitunti 11 - Agentista agenttiin (A2A) protokolla


## Asennus


In [ ]:
%pip install agent-framework azure-ai-projects azure-identity

In [ ]:
import os
import asyncio
from agent_framework import tool, AgentResponseUpdate, WorkflowBuilder
from agent_framework.azure import AzureAIProjectAgentProvider
from azure.identity import AzureCliCredential

In [ ]:

provider = AzureAIProjectAgentProvider(credential=AzureCliCredential())

## Mikä on A2A-protokolla?

The **Agent-to-Agent (A2A) protocol** is an open standard that enables AI agents to communicate,
discover each other, and collaborate — even when they are built on different frameworks or hosted
by different services.

Keskeiset käsitteet:

- **Löytäminen** – Agentit julkaisevat *agenttikortin*, joka kuvaa niiden kyvykkyyksiä, jolloin muiden agenttien (tai orkestroijien) on helppo löytää oikea asiantuntija tehtävään.
- **Viestinvälitys** – Agentit vaihtavat jäsenneltyjä viestejä yhteisen protokollan kautta, joten yhden agentin pyyntö voidaan ymmärtää ja toteuttaa toisella riippumatta sen sisäisestä toteutuksesta.
- **Tehtävän elinkaari** – A2A määrittelee tilat kuten *submitted*, *working*, *completed*, ja *failed*, antaen orkestroijalle täydellisen näkyvyyden siihen, miten delegoitu tehtävä etenee.

In this lesson we simulate A2A-style collaboration by wiring three specialized travel agents
into a workflow where each agent contributes its expertise and passes results to the next.


## Erikoistuneiden matka-agenttien luominen


In [ ]:
currency_agent = await provider.create_agent(
    name="CurrencyExchangeAgent",
    instructions="""You are a currency exchange specialist. You help travelers understand:
- Current exchange rates between currencies
- Best times to exchange money
- Tips for getting the best rates
When asked about a destination, provide relevant currency information.""",
)

activity_agent = await provider.create_agent(
    name="ActivityPlannerAgent",
    instructions="""You are a local activities specialist. You recommend:
- Must-see attractions and hidden gems
- Local experiences and cultural activities
- Restaurant and dining recommendations
Tailor suggestions to the traveler's interests.""",
)

travel_manager = await provider.create_agent(
    name="TravelManagerAgent",
    instructions="""You are a travel manager who coordinates between specialist agents.
When planning a trip:
1. Gather currency information from the currency specialist
2. Get activity recommendations from the activity planner
3. Synthesize everything into a cohesive travel brief
Present the final plan in an organized, easy-to-read format.""",
)

## Moni-agenttien yhteistyö työnkulun kautta

Yhdistämme kolme agenttia peräkkäiseksi työnkuluksi, joka jäljittelee A2A-viestinvälitystä:

1. **CurrencyExchangeAgent** vastaanottaa käyttäjän pyynnön ja tuottaa valuuttaneuvontaa.
2. **ActivityPlannerAgent** vastaanottaa rikastetun kontekstin ja lisää aktiviteettisuosituksia.
3. **TravelManagerAgent** yhdistää molemmat syötteet lopulliseksi matkakoosteeksi.


In [ ]:
workflow = WorkflowBuilder(start_executor=currency_agent) \
    .add_edge(currency_agent, activity_agent) \
    .add_edge(activity_agent, travel_manager) \
    .build()

last_author = None
events = workflow.run(
    "Plan a week-long trip to Tokyo. I love food, temples, and technology.",
    stream=True,
)
async for event in events:
    if event.type == "output" and isinstance(event.data, AgentResponseUpdate):
        update = event.data
        author = update.author_name
        if author != last_author:
            if last_author is not None:
                print()
            print(f"\n{'='*50}")
            print(f"🤖 {author}:")
            print(f"{'='*50}")
            last_author = author
        print(update.text, end="", flush=True)

## A2A:n ymmärtäminen tuotannossa

Tuotantoympäristössä A2A-protokolla avaa tehokkaita palveluiden välisiä skenaarioita:

| Ominaisuus | Kuvaus |
|---|---|
| **Kehysten välinen yhteentoimivuus** | Yhdellä kehyksellä rakennettu agentti voi delegoida tehtäviä toisella A2A-yhteensopivalla kehyksellä rakennetulle agentille, mikä mahdollistaa aidon organisaatiorajat ylittävän yhteentoimivuuden. |
| **Palvelurajat** | Agentit voivat toimia erillisissä mikropalveluissa, eri pilvialueilla tai jopa eri organisaatioissa, ja silti tehdä yhteistyötä saumattomasti. |
| **Dynaaminen löytäminen** | Orkestroija voi kysyä Agent Card -rekisteriä ajonaikaisesti löytääkseen parhaiten sopivan asiantuntijan tiettyyn alitehtävään. |
| **Streaming & push-ilmoitukset** | A2A tukee Server-Sent Events (SSE):ää reaaliaikaisten etenemispäivitysten ja pitkäkestoisten tehtävien push-ilmoitusten tarjoamiseen. |

Yllä rakentamamme työnkulku on tämän mallin yksinkertaistettu, prosessin sisäinen versio. Todellisessa
käyttöönotossa jokainen agentti tarjoaisi HTTP-päätepisteen, julkaisisi Agent Cardin ja kommunikoisi
A2A JSON-RPC -protokollan kautta.


## Yhteenveto

Tässä oppitunnissa opit:

1. **Mikä A2A-protokolla on** — avoin standardi agenttien väliselle löytämiselle, viestinnälle ja tehtävien hallinnalle.
2. **Miten luoda erikoistuneita agenteja** — valuutanvaihtoagentti, aktiviteettisuunnittelija-agentti ja matkahallinnan orkestroija.
3. **Miten kytkeä agentit työnkulkuun** — käyttäen `WorkflowBuilder`-luokkaa mallintamaan agenttien välistä peräkkäistä viestinvälitystä.
4. **Miten A2A toimii tuotannossa** — mahdollistamalla eri kehysten ja palveluiden välisen yhteistyön dynaamisen löydettävyyden ja suoratoistopäivitysten avulla.


---

<!-- CO-OP TRANSLATOR DISCLAIMER START -->
Vastuuvapauslauseke:
Tämä asiakirja on käännetty käyttämällä tekoälypohjaista käännöspalvelua Co-op Translator (https://github.com/Azure/co-op-translator). Vaikka pyrimme tarkkuuteen, huomioithan, että automaattisissa käännöksissä saattaa esiintyä virheitä tai epätarkkuuksia. Alkuperäistä asiakirjaa sen alkuperäisellä kielellä tulee pitää määräävänä lähteenä. Tärkeiden tietojen osalta suositellaan ammattimaisen kääntäjän tekemää käännöstä. Emme ole vastuussa tästä käännöksestä johtuvista väärinymmärryksistä tai virhetulkinnoista.
<!-- CO-OP TRANSLATOR DISCLAIMER END -->
